# Case Study : Social Media Analytics and Sentiment Analysis Platform using Apache Spark

### Q1. Spark Initialization and Data Loading (2 Marks)
 - Initialize Spark and load social media datasets.

In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
try:
    import pyspark
    print("✅ PySpark is installed.")
    print("PySpark Version:", pyspark.__version__)
except ImportError:
    print("❌ PySpark is NOT installed.")

✅ PySpark is installed.
PySpark Version: 4.1.2


Java Version Check for the JVM, Spark need JVM

In [3]:
import subprocess

result = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)

print(result.stderr)

java version "21.0.9" 2025-10-21 LTS
Java(TM) SE Runtime Environment (build 21.0.9+7-LTS-338)
Java HotSpot(TM) 64-Bit Server VM (build 21.0.9+7-LTS-338, mixed mode, sharing)



### Spark Session Creation

- SparkSession is the entry point to the Spark application. Almost every modern Spark feature (DataFrames, Spark SQL, reading/writing data, MLlib, etc.) is accessed through it. Once it is created, we can start using Spark.

In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Social Media Sentiment Analysis")
    .master("local[*]")
    .getOrCreate()
)

Spark Session check

In [5]:
# Application Name
print("Application Name:", spark.sparkContext.appName)

# Spark Version
print("Spark Version:", spark.version)

# Spark Context
print("Spark Context:", spark.sparkContext)

Application Name: Social Media Sentiment Analysis
Spark Version: 4.1.2
Spark Context: <SparkContext master=local[*] appName=Social Media Sentiment Analysis>


Loading Data Set

In [6]:
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true") # for multiline text data we have to use it, other wise the spark will check data line by line.
    .csv("data/sentimentdataset.csv")
)

Whenever a DataFrame looks wrong,
1. Check the raw file
   -     ↓
2. Check the delimiter
    -    ↓
3. Check whether there are quoted fields
    -    ↓
4. Check for multiline records
    -    ↓
5. Check the schema
    -    ↓
6. Then suspect Spark


Showing the first 20 rows

In [7]:
df.show()

+--------------------+--------------------+---------------+---------------+----------------+------------------+-------------------+-------------------+---------+--------+
|             Post ID|        Post Content|Sentiment Label|Number of Likes|Number of Shares|Number of Comments|User Follower Count| Post Date and Time|Post Type|Language|
+--------------------+--------------------+---------------+---------------+----------------+------------------+-------------------+-------------------+---------+--------+
|aa391375-7355-44b...|Word who nor cent...|        Neutral|            157|             243|                64|               4921|2024-01-10 00:14:21|    video|      fr|
|1c9ec98d-437a-48d...|Begin administrat...|       Positive|            166|              49|               121|                612|2024-02-03 00:20:11|    image|      es|
|170e5b5b-1d9a-4d0...|Thousand total si...|       Positive|            185|             224|               179|               9441|2024-07-25 14:

Printing Schema

In [8]:
df.printSchema()

root
 |-- Post ID: string (nullable = true)
 |-- Post Content: string (nullable = true)
 |-- Sentiment Label: string (nullable = true)
 |-- Number of Likes: integer (nullable = true)
 |-- Number of Shares: integer (nullable = true)
 |-- Number of Comments: integer (nullable = true)
 |-- User Follower Count: integer (nullable = true)
 |-- Post Date and Time: timestamp (nullable = true)
 |-- Post Type: string (nullable = true)
 |-- Language: string (nullable = true)



- Our dataset is already loaded as a Spark DataFrame, which internally uses Spark's distributed execution engine, including partitioning and parallel processing. Since DataFrames provide better optimization through the Catalyst Optimizer and are easier to use, there is no practical need to convert the dataset into an RDD. We are creating an RDD only to understand the concept because the case study requires it.

In modern Spark (Spark 2.x, 3.x, 4.x), a DataFrame is not literally just an RDD with a schema.

Internally it is represented as:

DataFrame
     │
Logical Plan
     │
Catalyst Optimizer
     │
Physical Plan
     │
RDDs (during execution)

So RDDs are still involved in execution, but the DataFrame goes through several optimization stages first.

That's why DataFrames are faster.

### Q2. RDD Implementation (3 Marks)
- Create RDDs and perform transformation and action operations

In [12]:
rdd = df.rdd

Verify RDD

In [10]:
print(type(rdd))

<class 'pyspark.core.rdd.RDD'>


First row

In [14]:
rdd.first()

Row(Post ID='aa391375-7355-44b7-bcbf-97fb4e5a2ba3', Post Content='Word who nor center everything better political. Various court realize arrive. Push with story understand easy least life.\nAlready forget major result condition. Relationship approach though million. Carry challenge carry.', Sentiment Label='Neutral', Number of Likes=157, Number of Shares=243, Number of Comments=64, User Follower Count=4921, Post Date and Time=datetime.datetime(2024, 1, 10, 0, 14, 21), Post Type='video', Language='fr')

Count of rows

In [15]:
rdd.count()

2000

Null Value removals

In [16]:
filtered_rdd = rdd.filter(
    lambda row: row["Post Content"] is not None
)

In [17]:
filtered_rdd.count()

2000

Text lowercasing

In [18]:
cleaned_rdd = filtered_rdd.map(
    lambda row: (
        row["Post ID"],
        row["Post Content"].strip().lower(),
        row["Sentiment Label"],
        row["Number of Likes"],
        row["Number of Shares"],
        row["Number of Comments"],
        row["User Follower Count"],
        row["Post Date and Time"],
        row["Post Type"],
        row["Language"]
    )
)

Checking

In [19]:
cleaned_rdd.take(3)

[('aa391375-7355-44b7-bcbf-97fb4e5a2ba3',
  'word who nor center everything better political. various court realize arrive. push with story understand easy least life.\nalready forget major result condition. relationship approach though million. carry challenge carry.',
  'Neutral',
  157,
  243,
  64,
  4921,
  datetime.datetime(2024, 1, 10, 0, 14, 21),
  'video',
  'fr'),
 ('1c9ec98d-437a-48d9-9cba-bd5ad853c59a',
  'begin administration population good president particularly. some study them side.\nhead when central should add new agent body. energy reach third place identify. seem nature financial box civil fill.\nwatch maybe subject. letter she follow theory write. indicate game perhaps.',
  'Positive',
  166,
  49,
  121,
  612,
  datetime.datetime(2024, 2, 3, 0, 20, 11),
  'image',
  'es'),
 ('170e5b5b-1d9a-4d02-a957-93c4dbb18908',
  'thousand total sign. agree product relationship several stop conference.\nallow as behind. the usually around.\nus business member. test cost more 

Positive counts

In [20]:
positive_posts = cleaned_rdd.filter(
    lambda row: row[2] == "Positive"
)

positive_posts.count()

643

Cleaned RDD

In [21]:
post_content_rdd = cleaned_rdd.map(
    lambda row: row[1]
)

post_content_rdd.take(5)

['word who nor center everything better political. various court realize arrive. push with story understand easy least life.\nalready forget major result condition. relationship approach though million. carry challenge carry.',
 'begin administration population good president particularly. some study them side.\nhead when central should add new agent body. energy reach third place identify. seem nature financial box civil fill.\nwatch maybe subject. letter she follow theory write. indicate game perhaps.',
 'thousand total sign. agree product relationship several stop conference.\nallow as behind. the usually around.\nus business member. test cost more many all various. difficult table reveal.',
 'individual from news third. oil forget them different account skin.\nwhite vote stop region building. try direction amount eight second amount support trial.\nkid rather always. real energy television majority hold meeting everybody.',
 'time adult letter see reduce. attention suddenly it. age

### Q3. Key-Value Operations and Persistence (2 Marks)
- Implement key-value operations, shuffle operations, and persistence

Sentiments and number of Likes

In [22]:
pair_rdd = df.rdd.map(
    lambda row: (row["Sentiment Label"], row["Number of Likes"])
)

pair_rdd.take(5)

[('Neutral', 157),
 ('Positive', 166),
 ('Positive', 185),
 ('Neutral', 851),
 ('Negative', 709)]

Count of each Sentiments

In [23]:
likes_by_sentiment = pair_rdd.reduceByKey(
    lambda x, y: x + y
)

likes_by_sentiment.collect()

[('Neutral', 333087), ('Positive', 314344), ('Negative', 359327)]

In [24]:
likes_by_sentiment.cache()

PythonRDD[26] at collect at C:\Users\vssuj\AppData\Local\Temp\ipykernel_14848\2475190387.py:5

In [25]:
print(likes_by_sentiment.getStorageLevel())

Memory Serialized 1x Replicated


In [26]:
likes_by_sentiment.collect()

[('Neutral', 333087), ('Positive', 314344), ('Negative', 359327)]

### 4th Question. Spark DataFrame Operations (3 Marks) 
- Perform DataFrame operations for social media analytics.

group by sentiments

In [27]:
df.groupBy("Sentiment Label").count().show()

+---------------+-----+
|Sentiment Label|count|
+---------------+-----+
|        Neutral|  682|
|       Positive|  643|
|       Negative|  675|
+---------------+-----+



group by post type

In [28]:
df.groupBy("Post Type") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

+---------+-----+
|Post Type|count|
+---------+-----+
|    image|  684|
|    video|  666|
|     text|  650|
+---------+-----+



top 10 most liked post

In [29]:
df.select(
    "Post ID",
    "Sentiment Label",
    "Number of Likes"
).orderBy(
    "Number of Likes",
    ascending=False
).show(10, truncate=False)

+------------------------------------+---------------+---------------+
|Post ID                             |Sentiment Label|Number of Likes|
+------------------------------------+---------------+---------------+
|d33f8eed-d482-4fbe-8838-0f988426090b|Positive       |1000           |
|1193ddea-41da-410d-9cdc-e7bf3408561a|Negative       |1000           |
|d7f961ff-c9e1-4d68-9f8d-4120f1794848|Negative       |998            |
|aa292b69-df09-439f-bcce-faf55b5699af|Neutral        |998            |
|f1b32616-6b8b-4ae5-848a-1d2347cb8aef|Positive       |998            |
|5bb8f06c-455d-42ab-9328-7a73c3731d07|Neutral        |997            |
|e9ce4b21-d482-41e2-a08b-8f3e77732ceb|Negative       |997            |
|12a57fd5-5354-4a87-bc0c-6b758583d38a|Neutral        |997            |
|60aead5d-7c2b-4e11-b26e-827cec2784e9|Positive       |997            |
|81eb352f-acc4-41c2-9214-0e5b58d0a654|Neutral        |996            |
+------------------------------------+---------------+---------------+
only s

group by language

In [30]:
df.groupBy("Language") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

+--------+-----+
|Language|count|
+--------+-----+
|      es|  431|
|      de|  424|
|      en|  421|
|      fr|  370|
|      zh|  354|
+--------+-----+



## Q5. Exploratory Data Analysis and Spark SQL (5 Marks)
- Perform EDA and execute Spark SQL queries to:
- Analyze sentiment distribution.
- Determine trending topics.
- Identify influential users.
- Analyze engagement metrics.
- Generate sentiment trend reports

In [31]:
df.createOrReplaceTempView("social_media")

In [32]:
spark.sql("""
SELECT
    `Sentiment Label`,
    COUNT(*) AS Total_Posts
FROM social_media
GROUP BY `Sentiment Label`
ORDER BY Total_Posts DESC
""").show()

+---------------+-----------+
|Sentiment Label|Total_Posts|
+---------------+-----------+
|        Neutral|        682|
|       Negative|        675|
|       Positive|        643|
+---------------+-----------+



In [33]:
spark.sql("""
SELECT
    AVG(`Number of Likes`) AS Avg_Likes,
    AVG(`Number of Shares`) AS Avg_Shares,
    AVG(`Number of Comments`) AS Avg_Comments
FROM social_media
""").show()

+---------+----------+------------+
|Avg_Likes|Avg_Shares|Avg_Comments|
+---------+----------+------------+
|  503.379|   248.485|     102.805|
+---------+----------+------------+



In [34]:
from pyspark.sql.functions import col

df.createOrReplaceTempView("social_media")

spark.sql("""
SELECT
    `Post ID`,
    `Post Content`,
    (`Number of Likes` + `Number of Shares` + `Number of Comments`) AS Engagement
FROM social_media
ORDER BY Engagement DESC
LIMIT 10
""").show(truncate=False)

+------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+
|Post ID                             |Post Content                                                                                                                                                                                                                                                                      |Engagement|
+------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+
|d8f0eff2-a3e8-4bce-a180-

In [35]:
spark.sql("""
SELECT
    DATE(`Post Date and Time`) AS Post_Date,
    `Sentiment Label`,
    COUNT(*) AS Total_Posts
FROM social_media
GROUP BY
    DATE(`Post Date and Time`),
    `Sentiment Label`
ORDER BY Post_Date
""").show(50, truncate=False)

+----------+---------------+-----------+
|Post_Date |Sentiment Label|Total_Posts|
+----------+---------------+-----------+
|2024-01-01|Positive       |1          |
|2024-01-01|Negative       |3          |
|2024-01-02|Neutral        |2          |
|2024-01-02|Positive       |2          |
|2024-01-03|Negative       |3          |
|2024-01-03|Positive       |6          |
|2024-01-03|Neutral        |2          |
|2024-01-04|Positive       |2          |
|2024-01-05|Negative       |3          |
|2024-01-05|Positive       |2          |
|2024-01-05|Neutral        |1          |
|2024-01-06|Neutral        |4          |
|2024-01-06|Positive       |5          |
|2024-01-06|Negative       |3          |
|2024-01-07|Positive       |1          |
|2024-01-07|Negative       |2          |
|2024-01-08|Neutral        |4          |
|2024-01-08|Positive       |3          |
|2024-01-08|Negative       |3          |
|2024-01-09|Positive       |2          |
|2024-01-09|Negative       |3          |
|2024-01-09|Neut

In [36]:
spark.sql("""
SELECT
    Language,
    COUNT(*) AS Total_Posts
FROM social_media
GROUP BY Language
ORDER BY Total_Posts DESC
""").show()

+--------+-----------+
|Language|Total_Posts|
+--------+-----------+
|      es|        431|
|      de|        424|
|      en|        421|
|      fr|        370|
|      zh|        354|
+--------+-----------+



## Q6. ETL Pipeline Development (3 Marks)
- Develop ETL pipelines for social media data processing.

In [37]:
df = spark.read.csv(
    "data/sentimentdataset.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

In [38]:
from pyspark.sql.functions import trim, lower, col

processed_df = (
    df
    .dropna(subset=["Post Content", "Sentiment Label"])
    .withColumn("Post Content", trim(col("Post Content")))
    .withColumn("Sentiment Label", lower(col("Sentiment Label")))
)

In [39]:
processed_df.show()

+--------------------+--------------------+---------------+---------------+----------------+------------------+-------------------+-------------------+---------+--------+
|             Post ID|        Post Content|Sentiment Label|Number of Likes|Number of Shares|Number of Comments|User Follower Count| Post Date and Time|Post Type|Language|
+--------------------+--------------------+---------------+---------------+----------------+------------------+-------------------+-------------------+---------+--------+
|aa391375-7355-44b...|Word who nor cent...|        neutral|            157|             243|                64|               4921|2024-01-10 00:14:21|    video|      fr|
|1c9ec98d-437a-48d...|Begin administrat...|       positive|            166|              49|               121|                612|2024-02-03 00:20:11|    image|      es|
|170e5b5b-1d9a-4d0...|Thousand total si...|       positive|            185|             224|               179|               9441|2024-07-25 14:

## Q7. Machine Learning/Deep Learning Implementation (2 Marks)
- Implement a sentiment classification or trend prediction model

In [40]:
from pyspark.ml.feature import Tokenizer
from pyspark.ml.feature import HashingTF
from pyspark.ml.feature import IDF
from pyspark.ml.feature import StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [41]:
ml_df = df.select("Post Content", "Sentiment Label")

In [42]:
ml_df = ml_df.dropna()

In [43]:
label_indexer = StringIndexer(
    inputCol="Sentiment Label",
    outputCol="label"
)

In [44]:
tokenizer = Tokenizer(
    inputCol="Post Content",
    outputCol="words"
)

In [45]:
from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(
    inputCol="words",
    outputCol="filtered_words"
)

In [46]:
hashingTF = HashingTF(
    inputCol="words",
    outputCol="features"
)

In [47]:
idf = IDF(
    inputCol="features",
    outputCol="tfidfFeatures"
)

In [48]:
lr = LogisticRegression(
    featuresCol="tfidfFeatures",
    labelCol="label"
)

In [49]:
pipeline = Pipeline(
    stages=[
        label_indexer,
        tokenizer,
        hashingTF,
        idf,
        lr
    ]
)

In [50]:
train_data, test_data = ml_df.randomSplit([0.8, 0.2], seed=42)

In [51]:
model = pipeline.fit(train_data)

In [52]:
predictions = model.transform(test_data)

In [53]:
predictions.select(
    "Post Content",
    "Sentiment Label",
    "prediction"
).show(10, truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+----------+
|Post Content                                                                                                                                                                                                                                                                      |Sentiment Label|prediction|
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+----------+
|About especially lot look.\nCompare money skill skin. Prepare seem generation. Determin

In [54]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)

print("Accuracy:", accuracy)

Accuracy: 0.3687150837988827


### Checking for the cause of low accuraccy

The implemented Spark ML pipeline is correct. However, the dataset appears to contain randomly generated text that has little semantic relationship with the assigned sentiment labels. Because the input features do not provide meaningful information for predicting the labels, the model achieves an accuracy close to random guessing (approximately 36.9% for three classes). This indicates that the dataset is not well-suited for training a sentiment classification model rather than an error in the implementation."

In [56]:
df.groupBy("Sentiment Label").count().show()

+---------------+-----+
|Sentiment Label|count|
+---------------+-----+
|        Neutral|  682|
|       Positive|  643|
|       Negative|  675|
+---------------+-----+



Not a problem of imbalance in dataset.

In [57]:
predictions.groupBy("prediction").count().show()

+----------+-----+
|prediction|count|
+----------+-----+
|       0.0|  120|
|       1.0|  119|
|       2.0|  119|
+----------+-----+



In [59]:
predictions.select(
    "Post Content",
    "Sentiment Label",
    "prediction"
).show(20, truncate=False)


+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+----------+
|Post Content                                                                                                                                                                                                                                                                         |Sentiment Label|prediction|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+----------+
|About especially lot look.\nCompare money skill skin. Prepare seem generation.

Not a problem of the model, its predicting well. not an issue there.

In [60]:
df.select("Post Content", "Sentiment Label").show(10, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+
|Post Content                                                                                                                                                                                                                                                                          |Sentiment Label|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+
|Word who nor center everything better political. Various court realize arrive. Push with story understand ea

The problem is this data is synthetic so it doest have any difference in positive and negative sentiment in post content.

In [61]:
df.printSchema()

root
 |-- Post ID: string (nullable = true)
 |-- Post Content: string (nullable = true)
 |-- Sentiment Label: string (nullable = true)
 |-- Number of Likes: integer (nullable = true)
 |-- Number of Shares: integer (nullable = true)
 |-- Number of Comments: integer (nullable = true)
 |-- User Follower Count: integer (nullable = true)
 |-- Post Date and Time: timestamp (nullable = true)
 |-- Post Type: string (nullable = true)
 |-- Language: string (nullable = true)



Cheking positive post contents

In [62]:
df.filter(df["Sentiment Label"] == "Positive") \
  .select("Post Content") \
  .show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Post Content                                                                                                                                                                                                                                                                          |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Begin administration population good president particularly. Some study them side.\nHead when central should add new agent body. Energy reach third place id

checking negative post contents.

In [63]:
df.filter(df["Sentiment Label"] == "Negative") \
  .select("Post Content") \
  .show(5, truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Post Content                                                                                                                                                                                                                                                                             |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Time adult letter see reduce. Attention suddenly it. Agency eye decade art friend value.\nPresident follow bring stay push practice air. Truth scie

**Conclusion**

The Spark ML sentiment classification pipeline was implemented correctly using text preprocessing, feature extraction, and Logistic Regression. However, the model achieved an accuracy of approximately **36.9%**, which is only slightly higher than random guessing for a three-class classification problem.

Further analysis of the dataset showed that the post content consists of synthetically generated text with limited semantic meaning and weak correspondence to the assigned sentiment labels. As a result, the textual features do not contain sufficient sentiment-related patterns for the model to learn effective decision boundaries. Since the primary limitation lies in the quality and suitability of the dataset rather than the machine learning pipeline, additional preprocessing techniques, feature engineering, or hyperparameter tuning did not lead to any significant improvement in performance.

Therefore, it was concluded that this dataset is not suitable for training and evaluating a sentiment analysis model. To obtain meaningful results, a dataset containing real-world text with clearly expressed positive, negative, and neutral sentiments will be used in the next phase of the project.


# Trying on new Dataset.

In [65]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Twitter Sentiment Analysis") \
    .getOrCreate()

df = spark.read.csv(
    "data/twitter_dataset.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

df = df.select("text", "sentiment")

## Data Exploration

In [66]:
df.show(5, truncate=False)

+---------------------------------------------------------------------------+---------+
|text                                                                       |sentiment|
+---------------------------------------------------------------------------+---------+
| I`d have responded, if I were going                                       |neutral  |
| Sooo SAD I will miss you here in San Diego!!!                             |negative |
|my boss is bullying me...                                                  |negative |
| what interview! leave me alone                                            |negative |
| Sons of ****, why couldn`t they put them on the releases we already bought|negative |
+---------------------------------------------------------------------------+---------+
only showing top 5 rows


In [67]:
df.printSchema()

root
 |-- text: string (nullable = true)
 |-- sentiment: string (nullable = true)



In [68]:
print("Total Records:", df.count())

Total Records: 27481


In [69]:
from pyspark.sql.functions import col, sum

df.select(
    sum(col("text").isNull().cast("int")).alias("Missing Text"),
    sum(col("sentiment").isNull().cast("int")).alias("Missing Sentiment")
).show()

+------------+-----------------+
|Missing Text|Missing Sentiment|
+------------+-----------------+
|           1|                0|
+------------+-----------------+



In [70]:
df.groupBy("sentiment").count().show()

+---------+-----+
|sentiment|count|
+---------+-----+
| positive| 8582|
|  neutral|11118|
| negative| 7781|
+---------+-----+




## Data Preprocessing

In [71]:
df = df.dropna(subset=["text"])

print("Records after removing null values:", df.count())

Records after removing null values: 27480


In [72]:
df.select(
    sum(col("text").isNull().cast("int")).alias("Missing Text"),
    sum(col("sentiment").isNull().cast("int")).alias("Missing Sentiment")
).show()

+------------+-----------------+
|Missing Text|Missing Sentiment|
+------------+-----------------+
|           0|                0|
+------------+-----------------+



String to Labels

In [73]:
from pyspark.ml.feature import StringIndexer

label_indexer = StringIndexer(
    inputCol="sentiment",
    outputCol="label"
)

In [77]:
indexed_df = label_indexer.fit(df).transform(df)

In [78]:
indexed_df.select(
    "sentiment",
    "label"
).show(10)

+---------+-----+
|sentiment|label|
+---------+-----+
|  neutral|  0.0|
| negative|  2.0|
| negative|  2.0|
| negative|  2.0|
| negative|  2.0|
|  neutral|  0.0|
| positive|  1.0|
|  neutral|  0.0|
|  neutral|  0.0|
| positive|  1.0|
+---------+-----+
only showing top 10 rows


In [79]:
indexed_df.select("sentiment", "label").show(10)

+---------+-----+
|sentiment|label|
+---------+-----+
|  neutral|  0.0|
| negative|  2.0|
| negative|  2.0|
| negative|  2.0|
| negative|  2.0|
|  neutral|  0.0|
| positive|  1.0|
|  neutral|  0.0|
|  neutral|  0.0|
| positive|  1.0|
+---------+-----+
only showing top 10 rows


Tokenization

In [80]:
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(
    inputCol="text",
    outputCol="words"
)

In [81]:
tokenized_df = tokenizer.transform(indexed_df)

In [82]:
tokenized_df.select(
    "text",
    "words"
).show(5, truncate=False)

+---------------------------------------------------------------------------+-------------------------------------------------------------------------------------------+
|text                                                                       |words                                                                                      |
+---------------------------------------------------------------------------+-------------------------------------------------------------------------------------------+
| I`d have responded, if I were going                                       |[, i`d, have, responded,, if, i, were, going]                                              |
| Sooo SAD I will miss you here in San Diego!!!                             |[, sooo, sad, i, will, miss, you, here, in, san, diego!!!]                                 |
|my boss is bullying me...                                                  |[my, boss, is, bullying, me...]                                          

Remove Stopwords

In [83]:
from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(
    inputCol="words",
    outputCol="filtered_words"
)

In [84]:
filtered_df = remover.transform(tokenized_df)

In [85]:
filtered_df.select(
    "words",
    "filtered_words"
).show(5, truncate=False)

+-------------------------------------------------------------------------------------------+---------------------------------------------------------+
|words                                                                                      |filtered_words                                           |
+-------------------------------------------------------------------------------------------+---------------------------------------------------------+
|[, i`d, have, responded,, if, i, were, going]                                              |[, i`d, responded,, going]                               |
|[, sooo, sad, i, will, miss, you, here, in, san, diego!!!]                                 |[, sooo, sad, miss, san, diego!!!]                       |
|[my, boss, is, bullying, me...]                                                            |[boss, bullying, me...]                                  |
|[, what, interview!, leave, me, alone]                                                 

Vectorization

In [101]:
from pyspark.ml.feature import CountVectorizer

cv = CountVectorizer(
    inputCol="filtered_words",
    outputCol="rawFeatures",
    vocabSize=20000,
    minDF=2
)

In [102]:
from pyspark.ml.feature import IDF

idf = IDF(
    inputCol="rawFeatures",
    outputCol="features"
)

vectorization functions are created calling and applying will be done by pipeline

Train test split

In [103]:
train_data, test_data = filtered_df.randomSplit([0.8, 0.2], seed=42)

print("Training Records:", train_data.count())
print("Testing Records :", test_data.count())

Training Records: 22012
Testing Records : 5468


Logistic regression

In [104]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100
)

Pipeline

In [105]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[
    cv,
    idf,
    lr
])

In [106]:
model = pipeline.fit(train_data)

Predictions 

In [107]:
predictions = model.transform(test_data)

In [108]:
predictions.select(
    "text",
    "sentiment",
    "prediction"
).show(10, truncate=False)

+---------------------------------------------------------------------------------------------------+---------+----------+
|text                                                                                               |sentiment|prediction|
+---------------------------------------------------------------------------------------------------+---------+----------+
|      You`ll be missed!!  Bring me back  a keychain!                                               |negative |1.0       |
|     Soooooooo What Happened To Power ForReal?                                                     |neutral  |0.0       |
|    #followfriday thank you so much. I`m so behind. Still at about half of what I had.             |positive |1.0       |
|    Lmao Yes its on the 27th  I get so excited lol                                                 |positive |1.0       |
|    Whn r u goin to Europe?                                                                        |neutral  |0.0       |
|    hope all is

Evaluvation

In [109]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)

print("Accuracy:", accuracy)

Accuracy: 0.5440746159473299


In [110]:
predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 1142|
|  0.0|       1.0|  541|
|  0.0|       2.0|  523|
|  1.0|       0.0|  474|
|  1.0|       1.0| 1024|
|  1.0|       2.0|  213|
|  2.0|       0.0|  512|
|  2.0|       1.0|  230|
|  2.0|       2.0|  809|
+-----+----------+-----+



In [111]:
precision = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
).evaluate(predictions)

recall = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(predictions)

f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
).evaluate(predictions)

print("Precision :", precision)
print("Recall    :", recall)
print("F1 Score  :", f1)

Precision : 0.5435407814503351
Recall    : 0.5440746159473299
F1 Score  : 0.5436332709707847
